# 🤖 Module 1: LLM API Foundations
**Agentic AI Engineering Program**

Every agent you build in this program, however sophisticated it ends up looking, is a loop around
one HTTP call: `POST /v1/messages`. Tool use, streaming, memory, sub-agents, evals, all of it is
a variation on that same request. Get comfortable with the request itself first, and the rest of
the program stops looking like magic and starts looking like plumbing you understand.

This module builds that comfort, roughly in the order you will need it: how the API represents a
conversation despite having no memory of its own, how to control what it generates, how to keep
it inside its context window, how to stream it, how to make it return structured data instead of
prose, and how to stop paying to reprocess the same tokens on every turn.


| Section | Content |
|---|---|
| 1.0 | Setup: client, model, offline mode |
| 1.1 | Messages API: roles and conversation state |
| 1.2 | Sampling and decoding controls |
| 1.3 | Tokens and the context window |
| 1.4 | Streaming |
| 1.5 | Structured output |
| 1.6 | Prompt caching |

Every lesson opens with a short **analogy** to build intuition, then **the problem it solves**,
**the pitfall that bites in production**, and **where you meet it in real agent frameworks**.
Then the code.

Lesson cells run with or without an API key: without one they print the exact JSON payload that
*would* be sent. Exercises never need a key: they run against recorded API objects and pure logic,
so a solution is deterministically right or wrong.

---
## 1.0 Setup: client, model, offline mode

### Lesson: Client construction and credential resolution

**The intuition.** Don't think of `Anthropic()` as needing one key. It's more like a keyring: the
SDK checks an env var, then an auth token, then a saved CLI login, then workload identity, in
that order, before it gives up. So an unset `ANTHROPIC_API_KEY` doesn't prove nothing will
authenticate; it might just mean the second or third pocket has the key. Same logic applies to
model ids. They're names you look up, not passwords you misremember and retype, and guessing
wrong gets you the same 404 whether or not your credentials were ever the issue.

**The problem.** Every agent you will build is a loop around one HTTP endpoint: `POST /v1/messages`.
Tool use, streaming, structured output and caching are all *parameters of that single endpoint*, not
separate APIs. Getting the client and the model id right once removes a whole class of confusing
failures later.

**Production pitfall.** Two classics. First, hardcoding `api_key="sk-ant-..."` in the source: the SDK
already resolves credentials in order: `ANTHROPIC_API_KEY` → `ANTHROPIC_AUTH_TOKEN` → an
`ant auth login` profile → workload identity federation. So an unset `ANTHROPIC_API_KEY` does *not*
mean "no credentials". Second, pinning a model id you half-remember, like `claude-opus-5-20260101`.
Current model ids are complete as they are; appending a date suffix returns a 404 that people
routinely misdiagnose as an auth problem.

**Where you meet it.** Every framework wraps this exact constructor: LangChain's `ChatAnthropic`,
LlamaIndex's `Anthropic` LLM, the Claude Agent SDK. They all bottom out at `messages.create`.

**For the curious.** Credential resolution isn't just described in the docs, it's real
MIT-licensed code you can read in an afternoon: [`anthropic-sdk-python`](https://github.com/anthropics/anthropic-sdk-python) on GitHub. Seeing the actual fallback
chain tends to demystify what otherwise feels like magic.

In [ ]:
# ── Install (run once) ───────────────────────────────────────
# pip install anthropic pydantic

import json, os, time, math, random, re
from typing import Any, Callable

# ── Model ids are complete as-is — never append a date suffix ─
MODEL       = "claude-opus-5"      # 1M context, 128K max output, $5 / $25 per 1M tokens
CHEAP_MODEL = "claude-haiku-4-5"   # 200K context, $1 / $5 — cheap sub-tasks, classification

# ── Client: credentials come from the environment, never the source ──
try:
    import anthropic
    CLIENT = anthropic.Anthropic()   # ANTHROPIC_API_KEY | ANTHROPIC_AUTH_TOKEN | `ant auth login`
except Exception as exc:
    CLIENT = None
    print(f"offline mode ({type(exc).__name__}): {exc}")

LIVE = CLIENT is not None
print("LIVE =", LIVE)


def call(**kwargs):
    "Send the request when a client exists; otherwise print the payload that would be sent."
    if not LIVE:
        print("[offline] POST /v1/messages")
        print(json.dumps(kwargs, indent=2, default=str)[:900])
        return None
    return CLIENT.messages.create(**kwargs)


def text_of(response) -> str:
    "response.content is a LIST of blocks (text, thinking, tool_use...). Never assume [0] is text."
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


# Demo calls in this notebook use a small max_tokens to keep the drill cheap.
# Production defaults: ~16000 non-streaming, ~64000 streaming (see 1.2 and 1.4).
DEMO_MAX_TOKENS = 300

---
## 1.1 Messages API: roles and conversation state

### Lesson: Roles, content blocks, and the stateless request

**The intuition.** Picture a brilliant consultant with no memory of any call before this one.
Every time you ring, you fax over the entire case file first, or they can't say a word. That's
roughly what the Messages API is doing. The model isn't forgetting between turns; there's nothing
to forget, because each request is judged purely on what's attached to it. And what you attach
usually isn't one clean note. It's more like an envelope with several separate items inside: a
client brief, a legal disclaimer, maybe a photo clipped to the back. That's why `content` is a
list of typed blocks instead of one paragraph of text.

**The problem.** The API is **stateless**: it stores nothing between calls. A "conversation" is an
illusion you maintain by resending the entire transcript every turn. Once that clicks, a whole family
of agent bugs becomes obvious: the model did not "forget", *you* did not send it.

The three roles carry different authority. `system` is operator instruction (highest trust, sits
outside the conversation), `user` is input, `assistant` is what the model produced. Keeping untrusted
data (retrieved documents, tool output, scraped pages) inside `user` content instead of folding it
into `system` is the whole basis of prompt-injection defence.

**Production pitfall.** Content is not a string, it is a **list of typed blocks**.
`response.content[0].text` breaks the moment the model emits a `thinking` or `tool_use` block first,
which is exactly what happens once you enable thinking or tools: as soon as the toy becomes an
agent. Always filter on `block.type`.

**Where you meet it.** The `messages` array is the universal interchange format of the whole
ecosystem: LangGraph's `state["messages"]`, the Claude Agent SDK transcript, and every agent trace
viewer you will ever open are rendering this same list.

**For the curious.** Prompt injection, the reason untrusted data belongs in `user` and never in
`system`, was named and has been catalogued in the wild since 2022 by Simon Willison, at [simonwillison.net/tags/prompt-injection](https://simonwillison.net/tags/prompt-injection/).
And if "no memory between calls" nags at you, you've basically rediscovered the premise of
*Memento*: the API is Leonard, and `messages[]` is his box of Polaroids.

In [ ]:
# ── The minimal request ──────────────────────────────────────
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    system="You are a terse assistant. Answer in one sentence.",
    messages=[{"role": "user", "content": "What is an agent loop?"}],
)
print(text_of(resp))

# ── content: string shorthand vs. explicit block list ────────
# These two user messages are strictly equivalent on the wire:
short_form = {"role": "user", "content": "Summarise this log."}
block_form = {"role": "user", "content": [{"type": "text", "text": "Summarise this log."}]}

# The block form becomes mandatory as soon as a message mixes types
# (text + image, text + document, text + tool_result).
multi_block = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Here is yesterday's error log:"},
        {"type": "text", "text": "2026-08-18 03:12 ERROR upstream timeout (x412)"},
        {"type": "text", "text": "What single change would cut this the most?"},
    ],
}

# ── system: string vs. block list ────────────────────────────
# The block list is what you need to attach cache_control (see 1.6).
system_cached = [
    {"type": "text", "text": "You are a support triage agent.",
     "cache_control": {"type": "ephemeral"}},
]

# ── Role rules the API enforces ──────────────────────────────
# 1. messages[0] must have role "user"
# 2. consecutive same-role messages are legal — the API merges them into one turn
# 3. role "system" INSIDE messages is a separate, model-gated feature (next lesson)
legal_merge = [
    {"role": "user", "content": "First thought."},
    {"role": "user", "content": "Second thought, same turn."},   # merged server-side
    {"role": "assistant", "content": "Got both."},
]
print(json.dumps({"system": system_cached, "messages": legal_merge}, indent=1)[:400])

# ── Reading a response defensively ───────────────────────────
# Every response carries: id, model, content (list of blocks), stop_reason, usage
if resp is not None:
    for block in resp.content:
        print("block.type =", block.type)
    print("stop_reason:", resp.stop_reason)
    print("usage      :", resp.usage.input_tokens, "in /", resp.usage.output_tokens, "out")

### Lesson: Multi-turn state, append the blocks, not the string

**The intuition.** Now hand that same consultant, next call, not their own notes back but your
paraphrase of what they said. You mention "the ticket you opened, #12," and they draw a blank,
because you gave them your summary of their sentence, not the ticket itself. That's what
`response.content[0].text` throws away: the reasoning and the tool request both live in blocks a
paraphrase drops. There's a smaller version of the same mistake, too. When you need to slip the
consultant a new instruction mid-meeting, something like "keep it terse from now on," you don't
repaint the sign on the door they read walking in. You pass them a note across the table instead.
That's the practical difference between rewriting top-level `system` and appending a `role:
system` message.

**The problem.** To continue a conversation you append the assistant turn to `messages` and resend
everything. *What* you append matters far more than it looks.

**Production pitfall.** This one line is the most common bug in hand-rolled agent loops:

```python
messages.append({"role": "assistant", "content": response.content[0].text})   # ❌
```

It discards every non-text block. With thinking enabled you drop the `thinking` blocks, so the model
loses its own reasoning chain across turns. With tools you drop the `tool_use` block, so the
`tool_result` you send next refers to an id the model never sees, producing a 400 that reads like a schema
error. With server-side compaction you drop the `compaction` block and silently lose the summarised
history, which shows up much later as an agent that mysteriously forgot the first half of its task.
Append `response.content` **whole**.

The second pitfall is *where* a mid-conversation instruction goes. Editing top-level `system` on turn
12 rewrites the prefix that sits ahead of all twelve turns, so every cached token is reprocessed at
full price (1.6 covers why). On Claude Opus 5 / Opus 4.8 / Fable 5 you instead append a
`{"role": "system"}` **message**: it lands *after* the cached history, and unlike operator text
smuggled into a user turn, it cannot be spoofed by anything that writes user-visible input.

**Where you meet it.** Anthropic's multi-turn examples, the SDK Tool Runner's internal loop, and
LangGraph's `add_messages` reducer all perform exactly this append-the-blocks dance.

In [ ]:
# ── A minimal, correct conversation manager ──────────────────
class Conversation:
    "Stateless API + client-side transcript. Appends BLOCKS, never a bare string."

    def __init__(self, model: str = MODEL, system=None):
        self.model    = model
        self.system   = system
        self.messages: list[dict] = []

    def send(self, user_message: str, **kwargs):
        self.messages.append({"role": "user", "content": user_message})
        response = call(
            model=self.model,
            max_tokens=kwargs.pop("max_tokens", DEMO_MAX_TOKENS),
            system=self.system,
            messages=self.messages,
            **kwargs,
        )
        if response is not None:
            # ✅ full block list — preserves thinking / tool_use / compaction blocks
            self.messages.append({"role": "assistant", "content": response.content})
        return response


chat = Conversation(system="You are a helpful assistant. Be brief.")
chat.send("My name is Adrien.")
chat.send("What is my name?")          # only answerable because turn 1 was resent
print("transcript:", len(chat.messages), "messages")

# ── Mid-conversation operator instruction (model-gated) ──────
# Cache-preserving: top-level `system` stays byte-identical.
history = [
    {"role": "user", "content": "Draft the incident summary."},
    {"role": "assistant", "content": "Here is a draft: ..."},
    {"role": "user", "content": "Now shorten it."},
]

payload = {
    "model": MODEL,
    "max_tokens": DEMO_MAX_TOKENS,
    "system": [{"type": "text", "text": "You are an SRE assistant.",
                "cache_control": {"type": "ephemeral"}}],
    "messages": history + [
        {"role": "system", "content": "Terse mode enabled — keep responses under 40 words."},
    ],
}

# Placement rules for a `role: "system"` message:
#   • cannot be messages[0]      (use top-level `system` for the initial prompt)
#   • must follow a user message (or an assistant turn ending in server-tool use)
#   • must be last, or be followed by an assistant turn
#   • unsupported models return 400 "role 'system' is not supported on this model"
try:
    resp2 = call(**payload)
except Exception as exc:                # anthropic.BadRequestError in practice
    print("fallback — inline the instruction in a user turn:", type(exc).__name__)

### 🏋️ Exercises 1.1

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Transcript builder that respects the role rules
# ════════════════════════════════════════════════════════
# Implement build_transcript(events) -> list[dict].
# `events` is a list of (role, content) tuples in arrival order.
# The returned messages list must satisfy the API rules:
#   • drop any leading assistant events (messages[0] must be "user")
#   • merge consecutive same-role events into ONE message whose content
#     becomes a list of text blocks: [{"type": "text", "text": ...}, ...]
#   • an unmerged event keeps the string shorthand

def build_transcript(events: list[tuple[str, str]]) -> list[dict]:
    pass  # to complete

events = [
    ("assistant", "stray opening"),
    ("user", "First thought."),
    ("user", "Second thought."),
    ("assistant", "Got both."),
    ("user", "Continue."),
]
for m in build_transcript(events):
    print(m["role"], "->", m["content"])
# Expected: user -> [{'type': 'text', ...}, {'type': 'text', ...}]
#           assistant -> Got both.
#           user -> Continue.

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# def build_transcript(events):
#     msgs = []
#     for role, content in events:
#         if not msgs and role != "user":
#             continue                                   # drop leading assistant turns
#         if msgs and msgs[-1]["role"] == role:
#             prev = msgs[-1]["content"]
#             if isinstance(prev, str):
#                 prev = [{"type": "text", "text": prev}]
#             prev.append({"type": "text", "text": content})
#             msgs[-1]["content"] = prev
#         else:
#             msgs.append({"role": role, "content": content})
#     return msgs

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Append an assistant turn without losing blocks
# ════════════════════════════════════════════════════════
# `response` below mimics an SDK response: .content is a list of block
# objects with a .type discriminator and type-specific attributes.
# Implement append_assistant_turn(messages, response): append the assistant
# turn in the form you must SEND BACK to the API —
#   • content = the full block list, serialised to wire dicts, order preserved
#   • thinking -> {"type": "thinking", "thinking": ..., "signature": ...}
#   • text     -> {"type": "text", "text": ...}
#   • tool_use -> {"type": "tool_use", "id": ..., "name": ..., "input": ...}
# Return the messages list.

class Block:
    "Stand-in for anthropic.types.*Block: attribute access, .type discriminator."
    def __init__(self, **kw):
        self.__dict__.update(kw)

response = type("Response", (), {})()
response.content = [
    Block(type="thinking", thinking="The user wants the weather.", signature="sig_abc"),
    Block(type="text", text="Let me look that up."),
    Block(type="tool_use", id="toolu_01", name="get_weather", input={"city": "Paris"}),
]
response.stop_reason = "tool_use"

def append_assistant_turn(messages: list[dict], response) -> list[dict]:
    pass  # to complete

messages = [{"role": "user", "content": "Weather in Paris?"}]
print(json.dumps(append_assistant_turn(messages, response), indent=1))
# Expected: 2 messages; the assistant one carries the 3 blocks in original order

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# BLOCK_FIELDS = {
#     "thinking": ("thinking", "signature"),
#     "text":     ("text",),
#     "tool_use": ("id", "name", "input"),
# }
# def append_assistant_turn(messages, response):
#     blocks = []
#     for b in response.content:
#         block = {"type": b.type}
#         for field in BLOCK_FIELDS.get(b.type, ()):
#             block[field] = getattr(b, field)
#         blocks.append(block)
#     messages.append({"role": "assistant", "content": blocks})
#     return messages

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Validate mid-conversation system messages
# ════════════════════════════════════════════════════════
# A {"role": "system"} entry inside messages[] is legal only if:
#   (a) it is not messages[0]
#   (b) the entry before it has role "user"
#   (c) it is the last entry, OR the entry after it has role "assistant"
# Implement validate_system_placement(messages) -> list[str], one string per
# offending index, formatted "i=<index>: <reason>" with reason in:
#   "cannot be first"
#   "must follow a user message"
#   "must be last or followed by an assistant turn"
# Report only the FIRST rule violated at a given index, checked in order (a),(b),(c).

def validate_system_placement(messages: list[dict]) -> list[str]:
    pass  # to complete

conv = [
    {"role": "system", "content": "boot"},            # (a) illegal — first
    {"role": "user", "content": "hi"},
    {"role": "system", "content": "terse mode"},      # legal — follows user, then assistant
    {"role": "assistant", "content": "ok"},
    {"role": "system", "content": "verbose mode"},    # (b) illegal — follows assistant
    {"role": "user", "content": "go"},
    {"role": "system", "content": "final"},           # legal — last entry
]
print(validate_system_placement(conv))
# Expected: ['i=0: cannot be first', 'i=4: must follow a user message']

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def validate_system_placement(messages):
#     errors = []
#     for i, m in enumerate(messages):
#         if m["role"] != "system":
#             continue
#         if i == 0:
#             errors.append("i=0: cannot be first")
#         elif messages[i - 1]["role"] != "user":
#             errors.append(f"i={i}: must follow a user message")
#         elif i < len(messages) - 1 and messages[i + 1]["role"] != "assistant":
#             errors.append(f"i={i}: must be last or followed by an assistant turn")
#     return errors

---
## 1.2 Sampling and decoding controls

### Lesson: What temperature and top-p actually do, and why current models removed them

**The intuition.** Rank a restaurant's whole menu from favorite to least favorite, then decide
how adventurous you feel tonight. Near `temperature` 0 you always order dish #1: safe, but the
same every night. Turn it up and you start seriously considering dishes near the bottom, the ones
you ranked low for a reason. `top_p` is adventurous in a gentler way. Instead of reshuffling the
whole menu, it says: only let me choose among dishes that together make up 80% of what I'd
plausibly enjoy, then pick among those. Once an agent is reasoning step by step and deciding
which tool to call, you really don't want a dice roll anywhere near that decision, which is part
of why current models dropped the sampling knobs in favor of `effort`: a dial for how hard the
model thinks, not how randomly it answers.

**The problem.** A model does not emit a token, it emits a distribution over ~200k tokens. Decoding
parameters reshape that distribution before a token is drawn: `temperature` flattens or sharpens it,
`top_p` (nucleus) truncates it to the smallest set of tokens whose cumulative probability reaches *p*,
`top_k` truncates to a fixed count. Understanding the mechanism is still worth your time: it is why
a "creative" agent wanders off-task and why a low-temperature one gets stuck in loops.

**Production pitfall.** The advice "set `temperature=0` for agents" is now actively wrong.
On Claude Opus 5, Opus 4.8, Opus 4.7, Sonnet 5 and Fable 5, `temperature`, `top_p` and `top_k` have
been **removed**: sending any of them returns a 400. They remain on Opus 4.6 / Sonnet 4.6 and older.
And even where accepted, `temperature=0` never bought determinism: batching, GPU non-associativity and
routing make identical requests diverge. Code that depended on "temperature 0 means reproducible" was
depending on a bug in its own mental model.

What replaced them: **adaptive thinking** (`thinking={"type": "adaptive"}`, on by default on Opus 5)
and **effort** (`output_config={"effort": ...}`, `low` → `max`, default `high`). Effort is the dial
that actually matters now: it controls reasoning depth and total token spend. `xhigh` is the sweet
spot for most coding and agentic work on Opus 5 / 4.7 / 4.8 and Sonnet 5; `low` is for sub-agents and
trivial classification.

**Where you meet it.** Any 2023–2024 agent tutorial, most YAML configs shipped with agent frameworks,
and every `LLMConfig(temperature=0.0)` dataclass in your codebase. Migrating a project to a current
model means auditing those first. They fail loudly, which is the good case.

**For the curious.** Nucleus sampling isn't folklore, it's a named, citable result: Holtzman et
al., ["The Curious Case of Neural Text Degeneration"](https://arxiv.org/abs/1904.09751) (ICLR 2020). It diagnosed why both greedy decoding and pure random sampling fail,
and it's the origin of the "cut the unreliable tail instead of reshaping the whole distribution"
idea `top_p` is built on.

In [ ]:
# ── The mechanism, in pure Python (no API needed) ────────────
def softmax(logits: list[float], temperature: float = 1.0) -> list[float]:
    "Temperature scales the logits BEFORE normalising: low T sharpens, high T flattens."
    scaled = [x / temperature for x in logits]
    m = max(scaled)                                  # subtract max for numerical stability
    exps = [math.exp(x - m) for x in scaled]
    total = sum(exps)
    return [e / total for e in exps]


logits = [3.2, 2.9, 1.1, 0.4, -0.8]                  # 5 candidate tokens
for t in (0.2, 1.0, 2.0):
    probs = softmax(logits, t)
    print(f"T={t:<4} -> " + " ".join(f"{p:.3f}" for p in probs))

# T=0.2 concentrates almost all mass on the top token (greedy-like, repetitive)
# T=2.0 spreads mass onto tokens the model considered unlikely (creative, off-task)

# ── The modern request: effort + adaptive thinking ───────────
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    # ❌ temperature=0.0,          -> 400 on Opus 5 / 4.8 / 4.7, Sonnet 5, Fable 5
    # ❌ top_p=0.9, top_k=40,      -> 400 on the same models
    thinking={"type": "adaptive", "display": "summarized"},   # default display is "omitted"
    output_config={"effort": "low"},                          # low | medium | high | xhigh | max
    messages=[{"role": "user", "content": "Classify: 'refund not received'. One word."}],
)
print(text_of(resp))

# Effort by workload — the practical mapping:
#   low    : classification, routing, sub-agents, cheap extraction
#   medium : routine transformation, summarisation
#   high   : default; intelligence-sensitive work
#   xhigh  : coding and long-horizon agentic tasks (best default on Opus 5 / Sonnet 5)
#   max    : correctness matters more than cost

### Lesson: Output control, `max_tokens`, `stop_sequences`, and reading `stop_reason`

**The intuition.** `stop_reason` is the answer to one question: why did this call end? Like a
real phone call, there's more than one honest answer. The person finished their sentence, that's
`end_turn`. Their battery died mid-word, that's `max_tokens`. They hit a word you'd agreed would
end the call, that's `stop_sequence`. They need to grab something before continuing, that's
`tool_use`. Or they just declined to answer, that's `refusal`. Nobody transcribes a call without
first checking which of these happened. Treat a battery death like a polite goodbye and you'll
parse half a thought as if it were the whole one.

**The problem.** You need a hard ceiling on generation (cost, latency, abuse) and a way to know *why*
generation stopped. Those are two different parameters and one field, and confusing them produces
silent data corruption rather than errors.

**Production pitfall.** `max_tokens` is an enforced cutoff the model is **not aware of**. It does not
"wrap up" as it approaches the limit. It is cut mid-token-stream. This is the number-one cause of
"the model returned invalid JSON": the JSON was fine, it was truncated at character 4096. The tell is
`stop_reason == "max_tokens"`, which is why you branch on `stop_reason` *before* parsing `content`.
Do not lowball it: default to ~16000 non-streaming and ~64000 streaming; the low values you see in
tutorials exist to make demos cheap.

The model-aware alternative is a **task budget** (`output_config.task_budget`, beta, min 20 000
tokens): the server injects a countdown the model can see, so it paces itself and lands the plane
instead of being guillotined. That is what you want for an agent loop, not a smaller `max_tokens`.

`stop_details` is populated **only** when `stop_reason == "refusal"`. It is `null` for every other
value, so guard before reading `.category`.

**Where you meet it.** Truncated tool arguments in module 2, half-written plans in module 3, and the
`finish_reason` column of every eval harness you build in module 7.

In [ ]:
# ── stop_reason: branch before you parse ─────────────────────
STOP_REASONS = {
    "end_turn":      "finished naturally — safe to parse",
    "max_tokens":    "TRUNCATED — output is incomplete, do not parse as JSON",
    "stop_sequence": "hit one of your stop_sequences (the sequence is not in the text)",
    "tool_use":      "wants a tool — execute it and send tool_result back (module 2)",
    "pause_turn":    "long-running server tool paused; resend to resume",
    "refusal":       "safety decline — stop_details.category explains which classifier",
}
for reason, meaning in STOP_REASONS.items():
    print(f"{reason:<14} {meaning}")


def handle(response):
    "The dispatch every production caller needs."
    if response is None:
        return None
    if response.stop_reason == "refusal":
        cat = response.stop_details.category if response.stop_details else None
        raise RuntimeError(f"refused ({cat})")
    if response.stop_reason == "max_tokens":
        raise ValueError("truncated — raise max_tokens or use a task budget")
    return text_of(response)


# ── stop_sequences: cut generation on a marker ───────────────
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    stop_sequences=["\n\nHuman:", "</answer>"],   # the matched sequence is NOT returned in the text
    messages=[{"role": "user", "content": "Write one line, then </answer>."}],
)
if resp is not None:
    print(repr(text_of(resp)), "| stop_reason:", resp.stop_reason,
          "| matched:", resp.stop_sequence)

# ── Task budget: the model-aware ceiling (beta) ──────────────
# Use streaming: a large max_tokens without streaming trips the SDK's timeout guard.
budgeted = {
    "model": MODEL,
    "max_tokens": 64000,
    "betas": ["task-budgets-2026-03-13"],
    "output_config": {"effort": "high",
                      "task_budget": {"type": "tokens", "total": 64000}},  # min 20000
    "messages": [{"role": "user", "content": "Refactor this module end to end."}],
}
print(json.dumps(budgeted["output_config"], indent=1))

### 🏋️ Exercises 1.2

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Nucleus (top-p) sampling from scratch
# ════════════════════════════════════════════════════════
# Implement the decoding step the API used to expose.
#
# nucleus_sample(logits, temperature, top_p, rng) -> (chosen_index, kept, probs)
#   1. apply temperature, then softmax          (reuse softmax() from the lesson)
#   2. sort indices by descending probability
#   3. keep the smallest prefix whose cumulative probability is >= top_p
#      (always keep at least one token)
#   4. renormalise the kept probabilities so they sum to 1
#   5. draw with inverse-CDF sampling using rng.random()
# Return: chosen index (into the ORIGINAL logits), the kept indices in
# descending-probability order, and the renormalised probs aligned with `kept`.

def nucleus_sample(logits: list[float], temperature: float, top_p: float, rng):
    pass  # to complete

logits = [3.2, 2.9, 1.1, 0.4, -0.8]
idx, kept, probs = nucleus_sample(logits, temperature=1.0, top_p=0.8, rng=random.Random(0))
print("kept :", kept)
print("probs:", [round(p, 4) for p in probs])
print("drawn:", idx)
# Expected: kept [0, 1], probs [0.5744, 0.4256], drawn 1

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def nucleus_sample(logits, temperature, top_p, rng):
#     probs = softmax(logits, temperature)
#     order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
#     kept, cumulative = [], 0.0
#     for i in order:
#         kept.append(i)
#         cumulative += probs[i]
#         if cumulative >= top_p:
#             break
#     total = sum(probs[i] for i in kept)
#     kept_probs = [probs[i] / total for i in kept]
#     r, acc = rng.random(), 0.0
#     for i, p in zip(kept, kept_probs):
#         acc += p
#         if r <= acc:
#             return i, kept, kept_probs
#     return kept[-1], kept, kept_probs

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Model-compatibility shim for request params
# ════════════════════════════════════════════════════════
# You maintain a gateway that receives legacy request kwargs and must make
# them valid for the target model instead of returning a 400.
# Implement normalise_request(model, **kwargs) -> (clean_kwargs, warnings)
# applying, in this order:
#   1. temperature / top_p / top_k: drop them if the model is in SAMPLING_REMOVED
#      -> warning "<param> removed on <model>"
#   2. thinking={"type": "enabled", "budget_tokens": N}: on SAMPLING_REMOVED models
#      rewrite to {"type": "adaptive"}   -> warning "budget_tokens removed on <model>"
#   3. output_config["effort"] == "xhigh" on a model not in XHIGH_MODELS:
#      downgrade to "high"               -> warning "xhigh unavailable on <model>"
#   4. thinking={"type": "disabled"} combined with effort in {"xhigh", "max"}:
#      this is a 400 on Opus 5 -> drop the thinking key entirely (adaptive is the
#      default there)                     -> warning "disabled thinking rejected at <effort>"
# clean_kwargs must not be the same object as the input dicts (no mutation of caller state).

SAMPLING_REMOVED = {"claude-opus-5", "claude-opus-4-8", "claude-opus-4-7",
                    "claude-sonnet-5", "claude-fable-5"}
XHIGH_MODELS     = {"claude-opus-5", "claude-opus-4-8", "claude-opus-4-7",
                    "claude-sonnet-5", "claude-fable-5"}

def normalise_request(model: str, **kwargs) -> tuple[dict, list[str]]:
    pass  # to complete

clean, warns = normalise_request(
    "claude-opus-5",
    temperature=0.0,
    top_p=0.9,
    thinking={"type": "enabled", "budget_tokens": 4096},
    output_config={"effort": "xhigh"},
    max_tokens=16000,
)
print(clean)
print(warns)
# Expected clean: {'thinking': {'type': 'adaptive'}, 'output_config': {'effort': 'xhigh'},
#                  'max_tokens': 16000}
# Expected warns: ['temperature removed on claude-opus-5', 'top_p removed on claude-opus-5',
#                  'budget_tokens removed on claude-opus-5']

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def normalise_request(model, **kwargs):
#     clean, warnings = dict(kwargs), []
#     for param in ("temperature", "top_p", "top_k"):
#         if param in clean and model in SAMPLING_REMOVED:
#             clean.pop(param)
#             warnings.append(f"{param} removed on {model}")
#     thinking = clean.get("thinking")
#     if isinstance(thinking, dict):
#         thinking = dict(thinking)
#         if thinking.get("type") == "enabled" and model in SAMPLING_REMOVED:
#             thinking = {"type": "adaptive"}
#             warnings.append(f"budget_tokens removed on {model}")
#         clean["thinking"] = thinking
#     oc = clean.get("output_config")
#     if isinstance(oc, dict):
#         oc = dict(oc)
#         if oc.get("effort") == "xhigh" and model not in XHIGH_MODELS:
#             oc["effort"] = "high"
#             warnings.append(f"xhigh unavailable on {model}")
#         clean["output_config"] = oc
#     effort = clean.get("output_config", {}).get("effort")
#     if clean.get("thinking", {}).get("type") == "disabled" and effort in {"xhigh", "max"}:
#         clean.pop("thinking")
#         warnings.append(f"disabled thinking rejected at {effort}")
#     return clean, warnings

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Simulate server-side stop handling
# ════════════════════════════════════════════════════════
# Reproduce how the server decides stop_reason, so you can unit-test callers
# without burning tokens.
# simulate_stop(text, stop_sequences, max_chars) -> dict with keys
#   "text", "stop_reason", "stop_sequence"
# Rules:
#   • a stop sequence cuts the text BEFORE the match; the sequence is never returned
#   • the earliest match wins; ties are broken by the order of stop_sequences
#   • truncation at max_chars competes with the stop sequence — whichever
#     boundary comes first wins
#   • truncation -> stop_reason "max_tokens", stop_sequence None
#   • stop hit   -> stop_reason "stop_sequence", stop_sequence = the matched string
#   • neither    -> stop_reason "end_turn", stop_sequence None
#   • empty stop_sequences list is legal

def simulate_stop(text: str, stop_sequences: list[str], max_chars: int) -> dict:
    pass  # to complete

body = "Answer: 42.\n\nHuman: and then?</answer> trailing"
print(simulate_stop(body, ["\n\nHuman:", "</answer>"], max_chars=200))
print(simulate_stop(body, ["\n\nHuman:", "</answer>"], max_chars=8))
print(simulate_stop("short and clean", [], max_chars=200))
# Expected:
# {'text': 'Answer: 42.', 'stop_reason': 'stop_sequence', 'stop_sequence': '\n\nHuman:'}
# {'text': 'Answer: ', 'stop_reason': 'max_tokens', 'stop_sequence': None}
# {'text': 'short and clean', 'stop_reason': 'end_turn', 'stop_sequence': None}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def simulate_stop(text, stop_sequences, max_chars):
#     best_pos, best_seq = None, None
#     for seq in stop_sequences:
#         pos = text.find(seq)
#         if pos != -1 and (best_pos is None or pos < best_pos):
#             best_pos, best_seq = pos, seq
#     if best_pos is not None and best_pos <= max_chars:
#         return {"text": text[:best_pos], "stop_reason": "stop_sequence",
#                 "stop_sequence": best_seq}
#     if len(text) > max_chars:
#         return {"text": text[:max_chars], "stop_reason": "max_tokens", "stop_sequence": None}
#     return {"text": text, "stop_reason": "end_turn", "stop_sequence": None}

---
## 1.3 Tokens and the context window

### Lesson: Counting tokens correctly

**The intuition.** A token is less like a word and more like a slice of bread, and different
tokenizers slice the same loaf differently. `tiktoken` cuts bread the way OpenAI's models like
it. Claude's tokenizer cuts the same text into a different number of pieces, usually more of
them. Measuring Claude's context window in `tiktoken` slices is like judging whether a cake fits
in a box by counting some other bakery's slices. The number feels precise. It's also quietly
wrong, and you won't find out until the day the box is actually full.

**The problem.** Tokens are the unit of cost, of latency, and of the context limit. Every budgeting
decision an agent makes (how much history to keep, how many retrieved chunks to attach, whether to
summarise) is arithmetic on token counts. If your counts are wrong, your budgets are wrong.

**Production pitfall.** Using `tiktoken`. It is OpenAI's tokenizer: it undercounts Claude tokens by
roughly 15–20% on ordinary prose and far more on code, JSON and non-English text. A pipeline that
budgets 180k "tokens" with tiktoken and sends it to a 200k-context model will overflow in production
on exactly the inputs that matter. The `chars / 4` heuristic is worse. Use
`client.messages.count_tokens()`: same request shape as `messages.create`, returns `input_tokens`,
and counts are **model-specific** (pass the model you will actually call: the Opus 4.7-generation
tokenizer produces ~1×–1.35× the counts of the older one).

**Where you meet it.** Chunk sizing in module 4's RAG pipeline, the compaction trigger in module 4,
the per-run cost ledger in module 6. Get the counter right here and those all inherit it.

**For the curious.** [`tiktoken`](https://github.com/openai/tiktoken) is real and open source, and it's exactly as good at counting
Claude's tokens as a French dictionary is at counting English syllables: it counts something,
just not the thing you asked about. For the real thing, [Anthropic's token counting docs](https://platform.claude.com/docs/en/build-with-claude/token-counting) cover images, PDFs, and thinking
blocks too.

In [ ]:
# ── Count before you send ────────────────────────────────────
document = "Retrieval augmented generation " * 40

if LIVE:
    counted = CLIENT.messages.count_tokens(
        model=MODEL,
        system="You are a summariser.",
        messages=[{"role": "user", "content": document}],
    )
    print("input_tokens:", counted.input_tokens)
else:
    print("[offline] POST /v1/messages/count_tokens — same body shape as messages.create")


def rough_tokens(text: str) -> int:
    "Offline stand-in ONLY. Never ship this: it is wrong for code, JSON and non-English."
    return max(1, len(text) // 4)


print("rough estimate:", rough_tokens(document), "tokens (indicative, not a budget)")

# ── Context and output limits per model ──────────────────────
LIMITS = {
    "claude-opus-5":     {"context": 1_000_000, "max_output": 128_000},
    "claude-sonnet-5":   {"context": 1_000_000, "max_output": 128_000},
    "claude-haiku-4-5":  {"context":   200_000, "max_output":   8_192},
}
for name, lim in LIMITS.items():
    print(f"{name:<20} context={lim['context']:>9,}  max_output={lim['max_output']:>7,}")

# The context window covers the WHOLE request: system + tools + every message + the
# reply being generated. Budget for max_tokens too, not just what you send.

### Lesson: Usage accounting and keeping history inside the window

**The intuition.** `usage.input_tokens` on its own is a receipt that only lists what you paid
full price for today. It leaves off everything that came out of the pantry, which is the cache,
so reading just that number leaves you thinking you're running a kitchen for four when you're
actually cooking for forty. Trimming old messages to save room has its own version of this trap.
It's like editing a play down to its last few pages: cut carelessly and an actor is left on stage
reacting to a prop, "the wrench I asked you to check," that got cut from an earlier scene. The
API notices immediately, because `tool_use` and `tool_result` are a matched pair.

**The problem.** Every response carries a `usage` object, and it is the only honest signal you have
about what a run actually cost. Agent loops resend the full transcript each turn, so token spend grows
quadratically with turn count. The run that costs $0.04 in your notebook costs $40 in production for
reasons that are entirely visible in `usage`, if you read it correctly.

**Production pitfall.** `usage.input_tokens` is **not** the prompt size. It is the *uncached
remainder*. The real prompt is `input_tokens + cache_creation_input_tokens + cache_read_input_tokens`.
Teams routinely report "our agent only sends 4k tokens" while actually sending 200k, 196k of it served
from cache. The second pitfall is naïve trimming: dropping the oldest messages until you fit will
happily orphan an `assistant` turn at index 0 (the API requires `messages[0]` to be `user`) or split a
`tool_use` from its `tool_result`, producing a 400 that only fires on long conversations, i.e. in production.

Three ways to stay inside the window, in increasing order of sophistication: a **sliding window**
(cheap, forgetful), **context editing** (`context_management.edits` with `clear_tool_uses_20250919`,
which *clears* old tool results), and **compaction** (`compact_20260112`, which *summarises* earlier context
server-side and returns a compaction block you must send back). Module 4 builds all three by hand.

**Where you meet it.** The cost dashboard in module 6, and the "why did my agent forget?" bug report
that turns out to be a trimmer that dropped the wrong turn.

In [ ]:
# ── Reading usage correctly ──────────────────────────────────
usage = {
    "input_tokens": 1_200,                  # billed at full price
    "cache_creation_input_tokens": 8_000,   # written to cache this turn (~1.25x price)
    "cache_read_input_tokens": 190_000,     # served from cache (~0.1x price)
    "output_tokens": 640,
}
prompt_size = (usage["input_tokens"]
               + usage["cache_creation_input_tokens"]
               + usage["cache_read_input_tokens"])
print(f"actual prompt: {prompt_size:,} tokens (input_tokens alone showed {usage['input_tokens']:,})")

# ── Pricing, $ per 1M tokens ─────────────────────────────────
PRICES = {
    "claude-opus-5":    {"input": 5.00, "output": 25.00},
    "claude-sonnet-5":  {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5": {"input": 1.00, "output":  5.00},
}
CACHE_WRITE_MULT = {"5m": 1.25, "1h": 2.00}   # premium paid once, on write
CACHE_READ_MULT  = 0.10                       # cache reads cost ~10% of base input

p = PRICES[MODEL]
cost = (usage["input_tokens"] * p["input"]
        + usage["cache_creation_input_tokens"] * p["input"] * CACHE_WRITE_MULT["5m"]
        + usage["cache_read_input_tokens"] * p["input"] * CACHE_READ_MULT
        + usage["output_tokens"] * p["output"]) / 1_000_000
print(f"turn cost: ${cost:.4f}  (uncached, that prompt would be "
      f"${prompt_size * p['input'] / 1_000_000:.4f} in input alone)")

# ── Sliding window that respects the API's structural rules ──
def sliding_window(messages: list[dict], keep_last: int) -> list[dict]:
    "Keep the tail, then repair the head so messages[0] is a user turn."
    kept = messages[-keep_last:]
    while kept and kept[0]["role"] != "user":
        kept = kept[1:]
    return kept


convo = [{"role": r, "content": f"turn {i}"}
         for i, r in enumerate(["user", "assistant"] * 5)]
print("kept roles:", [m["role"] for m in sliding_window(convo, keep_last=5)])

# ── Server-side alternatives (module 4 goes deeper) ──────────
context_managed = {
    "betas": ["context-management-2025-06-27"],
    "context_management": {"edits": [{"type": "clear_tool_uses_20250919"}]},   # clears
}
compacted = {
    "betas": ["compact-2026-01-12"],
    "context_management": {"edits": [{"type": "compact_20260112"}]},           # summarises
}
print(json.dumps([context_managed, compacted], indent=1)[:300])

### 🏋️ Exercises 1.3

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Cost ledger for an agent run
# ════════════════════════════════════════════════════════
# Implement run_cost(usages, model) -> dict with keys
#   "input", "cache_write", "cache_read", "output", "total", "prompt_tokens"
# where each value is a float in dollars (round to 6 decimals) except
# "prompt_tokens", the integer sum over all turns of
#   input_tokens + cache_creation_input_tokens + cache_read_input_tokens.
# Pricing: PRICES[model] in $/1M; cache writes cost input_price * 1.25 for a
# "5m" ttl and * 2.00 for "1h"; cache reads cost input_price * 0.10.
# Each usage dict may carry "ttl" (default "5m").

usages = [
    {"input_tokens": 12_000, "cache_creation_input_tokens": 30_000,
     "cache_read_input_tokens": 0, "output_tokens": 800},
    {"input_tokens": 400, "cache_creation_input_tokens": 0,
     "cache_read_input_tokens": 30_000, "output_tokens": 1_200},
    {"input_tokens": 350, "cache_creation_input_tokens": 5_000,
     "cache_read_input_tokens": 30_000, "output_tokens": 900, "ttl": "1h"},
]

def run_cost(usages: list[dict], model: str) -> dict:
    pass  # to complete

print(run_cost(usages, "claude-opus-5"))
# Expected: {'input': 0.06375, 'cache_write': 0.2375, 'cache_read': 0.03,
#            'output': 0.0725, 'total': 0.40375, 'prompt_tokens': 107750}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def run_cost(usages, model):
#     p = PRICES[model]
#     acc = {"input": 0.0, "cache_write": 0.0, "cache_read": 0.0, "output": 0.0}
#     prompt_tokens = 0
#     for u in usages:
#         mult = CACHE_WRITE_MULT[u.get("ttl", "5m")]
#         acc["input"]       += u["input_tokens"] * p["input"]
#         acc["cache_write"] += u["cache_creation_input_tokens"] * p["input"] * mult
#         acc["cache_read"]  += u["cache_read_input_tokens"] * p["input"] * CACHE_READ_MULT
#         acc["output"]      += u["output_tokens"] * p["output"]
#         prompt_tokens += (u["input_tokens"] + u["cache_creation_input_tokens"]
#                           + u["cache_read_input_tokens"])
#     out = {k: round(v / 1_000_000, 6) for k, v in acc.items()}
#     out["total"] = round(sum(out.values()), 6)
#     out["prompt_tokens"] = prompt_tokens
#     return out

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Fit a transcript into a token budget
# ════════════════════════════════════════════════════════
# Implement fit_history(messages, budget) -> list[dict].
# Each message is {"role": ..., "tokens": int}.
# Rules, in order:
#   • walk from the END backwards, keeping messages while the running total
#     stays <= budget
#   • the last message is ALWAYS kept, even if it alone exceeds the budget
#   • the result must start with a "user" message: drop leading non-user
#     messages from the kept slice afterwards
#   • preserve the original order
# Return the kept messages.

history = [
    {"role": "user",      "tokens": 500},
    {"role": "assistant", "tokens": 900},
    {"role": "user",      "tokens": 300},
    {"role": "assistant", "tokens": 700},
    {"role": "user",      "tokens": 200},
]

def fit_history(messages: list[dict], budget: int) -> list[dict]:
    pass  # to complete

print([(m["role"], m["tokens"]) for m in fit_history(history, budget=1300)])
print([(m["role"], m["tokens"]) for m in fit_history(history, budget=100)])
# Expected: [('user', 300), ('assistant', 700), ('user', 200)]
#           [('user', 200)]

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def fit_history(messages, budget):
#     kept, total = [], 0
#     for m in reversed(messages):
#         if kept and total + m["tokens"] > budget:
#             break
#         kept.append(m)
#         total += m["tokens"]
#     kept.reverse()
#     while len(kept) > 1 and kept[0]["role"] != "user":
#         kept.pop(0)
#     return kept

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Token-aware chunking with overlap
# ════════════════════════════════════════════════════════
# Implement chunk_by_tokens(text, count_fn, max_tokens, overlap_tokens).
# `count_fn(str) -> int` is INJECTED so production can pass
# client.messages.count_tokens while tests pass a deterministic stub.
# Rules:
#   • split the text on whitespace into words
#   • greedily pack words into a chunk while count_fn(chunk) <= max_tokens
#   • the next chunk restarts with the trailing words of the previous chunk
#     whose count_fn is <= overlap_tokens (take as many trailing words as fit)
#   • a single word longer than max_tokens still becomes its own chunk
#   • return the list of chunk strings; overlap_tokens must be < max_tokens

def chunk_by_tokens(text: str, count_fn, max_tokens: int, overlap_tokens: int) -> list[str]:
    pass  # to complete

words = " ".join(f"w{i}" for i in range(10))
by_word = lambda s: len(s.split())        # stub counter: 1 token per word
for c in chunk_by_tokens(words, by_word, max_tokens=4, overlap_tokens=1):
    print(c)
# Expected: w0 w1 w2 w3
#           w3 w4 w5 w6
#           w6 w7 w8 w9

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def chunk_by_tokens(text, count_fn, max_tokens, overlap_tokens):
#     assert overlap_tokens < max_tokens, "overlap must be smaller than the chunk"
#     words, chunks, current = text.split(), [], []
#     for word in words:
#         candidate = current + [word]
#         if current and count_fn(" ".join(candidate)) > max_tokens:
#             chunks.append(" ".join(current))
#             tail = []
#             for w in reversed(current):
#                 if count_fn(" ".join([w] + tail)) > overlap_tokens:
#                     break
#                 tail.insert(0, w)
#             current = tail + [word]
#         else:
#             current = candidate
#     if current:
#         chunks.append(" ".join(current))
#     return chunks

---
## 1.4 Streaming

### Lesson: The event protocol

**The intuition.** A non-streaming call is a letter that shows up in your mailbox only once it's
fully written and sealed. Silence, then everything at once. Streaming is watching through the
window while it's written, word by word, live. Here's the part people miss: the writer may be
filling several notebooks at the same time, a scratch notebook for reasoning, a clean one for the
reply, an order form for a tool call, each one marked with an index. What actually arrives on the
wire reads more like "notebook 2, new page," "notebook 2, new word," "notebook 2, page closed."
Mix pages from different notebooks into one string and you've glued the model's private reasoning
onto the front of its answer.

**The problem.** A non-streaming request returns nothing until the last token is generated. For a
2000-token answer that is 20–40 seconds of silence, unacceptable in a chat UI and, worse, a
connection that idle proxies happily kill. Streaming turns one long request into a sequence of
server-sent events so you can render the first token in a few hundred milliseconds and keep the
socket alive.

**Production pitfall.** Streaming is not just a UX nicety, it is a **timeout requirement**: the SDK
refuses a non-streaming request whose `max_tokens` it estimates will run past ~10 minutes, raising a
`ValueError` before any HTTP call. Since current models allow up to 128k output tokens, any generous
`max_tokens` forces you into `.stream()`. The other trap is assuming events arrive for one block:
a single response interleaves `thinking`, `text` and `tool_use` blocks, each with its own
`content_block_start` / `delta` / `stop` triplet, addressed by `index`. Code that concatenates every
delta into one string silently merges the model's reasoning into its answer.

**Where you meet it.** The SSE endpoint you build in module 6, the token-by-token rendering in every
chat product, and the "why is my first byte 30 s late" incident that turns out to be a non-streaming
call behind a load balancer with a 30 s idle timeout.

**For the curious.** Server-sent events are a real, boring, two-decade-old web standard, not an
Anthropic invention: [MDN's guide](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) covers the wire format itself, and [Anthropic's streaming docs](https://platform.claude.com/docs/en/build-with-claude/streaming) show the exact event
sequence per SDK.

In [ ]:
# ── The helper: accumulates state for you ────────────────────
if LIVE:
    with CLIENT.messages.stream(
        model=MODEL,
        max_tokens=DEMO_MAX_TOKENS,
        messages=[{"role": "user", "content": "Name three agent failure modes."}],
    ) as stream:
        for chunk in stream.text_stream:          # text deltas only, already filtered
            print(chunk, end="", flush=True)
        final = stream.get_final_message()        # full Message once the stream ends
    print("\n\nstop_reason:", final.stop_reason, "| output:", final.usage.output_tokens)
else:
    print("[offline] client.messages.stream(...) -> text_stream / get_final_message()")

# ── The raw event loop: what text_stream hides ───────────────
EVENT_SEQUENCE = [
    ("message_start",        "message metadata + input usage — fires once"),
    ("content_block_start",  "a block begins; content_block.type says which kind"),
    ("content_block_delta",  "incremental payload; delta.type varies by block"),
    ("content_block_stop",   "the block is complete — only now is its JSON parseable"),
    ("message_delta",        "top-level updates: stop_reason and FINAL output_tokens"),
    ("message_stop",         "end of stream"),
]
for name, meaning in EVENT_SEQUENCE:
    print(f"{name:<20} {meaning}")

# delta.type by block type:
#   text      -> text_delta        (.text)
#   thinking  -> thinking_delta    (.thinking)     — empty unless display="summarized"
#   tool_use  -> input_json_delta  (.partial_json) — a JSON FRAGMENT, not JSON

### Lesson: Consuming a stream safely

**The intuition.** Reading `partial_json` mid-stream is like trying to read a fax while it's
still sliding out of the machine. The line forming right now might say "do not," and three
seconds later it says "do not forget." Waiting for `content_block_stop` before you parse is just
waiting for the page to finish printing. And if the fax jams halfway, say the connection drops,
you don't staple the half page into the permanent file and call it the memo. You ask them to send
it again.

**The problem.** A stream gives you partial state. Deciding what is safe to act on at each moment is
the entire skill: text can be rendered as it arrives, tool arguments absolutely cannot.

**Production pitfall.** `input_json_delta.partial_json` carries *fragments*: `{"ci`, `ty": "Par`,
`is"}`. Calling `json.loads` on the buffer mid-block throws on every fragment but the last, so people
wrap it in a bare `except: pass` and then wonder why a tool occasionally fires with half its
arguments. The rule: accumulate fragments, parse **only** at `content_block_stop`. The second trap is
usage accounting: `message_start` reports input tokens with `output_tokens: 1`, but the real output count
arrives in `message_delta` at the end. Reading it from `message_start` under-reports every run.

Two more that bite: a mid-stream disconnection leaves you holding a *partial* assistant turn (never
append it to history as if it were complete, or you would teach the model that truncated answers are
normal), and on Opus 5 / 4.8 / 4.7 / Fable 5 the default `thinking.display` is `"omitted"`, so
`thinking_delta` events arrive with empty text and the UI shows a long pause. Set
`display: "summarized"` when you stream reasoning to users.

**Where you meet it.** Module 2's tool loop consumes exactly these events; module 6 forwards them
over SSE to a browser; module 7 asks you to write this accumulator under time pressure.

In [ ]:
# ── Manual event handling: one accumulator per block index ───
if LIVE:
    blocks: dict[int, dict] = {}
    with CLIENT.messages.stream(
        model=MODEL,
        max_tokens=DEMO_MAX_TOKENS,
        thinking={"type": "adaptive", "display": "summarized"},
        messages=[{"role": "user", "content": "Two sentences on retry storms."}],
    ) as stream:
        for event in stream:
            if event.type == "content_block_start":
                blocks[event.index] = {"type": event.content_block.type, "buf": []}
            elif event.type == "content_block_delta":
                d = event.delta
                if d.type == "text_delta":
                    blocks[event.index]["buf"].append(d.text)
                elif d.type == "thinking_delta":
                    blocks[event.index]["buf"].append(d.thinking)
                elif d.type == "input_json_delta":
                    blocks[event.index]["buf"].append(d.partial_json)   # fragment!
            elif event.type == "content_block_stop":
                b = blocks[event.index]
                b["value"] = "".join(b["buf"])
                if b["type"] == "tool_use":
                    b["value"] = json.loads(b["value"])   # safe ONLY here
    print({i: b["type"] for i, b in blocks.items()})
else:
    print("[offline] see the recorded event fixture in the exercises below")

# ── Time to first token, measured properly ───────────────────
def measure_ttft(client, **kwargs) -> tuple[float, float]:
    "Returns (ttft_seconds, total_seconds). TTFT is what users perceive as latency."
    start = time.perf_counter()
    first = None
    with client.messages.stream(**kwargs) as stream:
        for _ in stream.text_stream:
            if first is None:
                first = time.perf_counter() - start
    return first or 0.0, time.perf_counter() - start


# ── An interrupted stream is NOT a turn ──────────────────────
# On APIConnectionError mid-stream you hold partial content. Retry the turn;
# do not append the fragment to `messages` as an assistant turn.

### 🏋️ Exercises 1.4

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Rebuild the final message from raw events
# ════════════════════════════════════════════════════════
# RECORDED is a faithful capture of the SSE events for one response
# (dicts here; the SDK gives objects with the same field names).
# Implement accumulate(events) -> dict with keys "id", "model", "content",
# "stop_reason", "usage", reconstructing exactly what a non-streaming call
# would have returned:
#   • content: blocks in index order; text blocks -> {"type","text"},
#     tool_use blocks -> {"type","id","name","input"} with input PARSED
#     from the concatenated partial_json fragments
#   • stop_reason: from message_delta
#   • usage: {"input_tokens": from message_start, "output_tokens": from message_delta}

RECORDED = [
    {"type": "message_start", "message": {
        "id": "msg_017x", "model": "claude-opus-5", "role": "assistant",
        "content": [], "stop_reason": None,
        "usage": {"input_tokens": 1450, "output_tokens": 1}}},
    {"type": "content_block_start", "index": 0,
     "content_block": {"type": "text", "text": ""}},
    {"type": "content_block_delta", "index": 0,
     "delta": {"type": "text_delta", "text": "Let me check "}},
    {"type": "content_block_delta", "index": 0,
     "delta": {"type": "text_delta", "text": "the weather."}},
    {"type": "content_block_stop", "index": 0},
    {"type": "content_block_start", "index": 1,
     "content_block": {"type": "tool_use", "id": "toolu_09", "name": "get_weather", "input": {}}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": '{"ci'}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": 'ty": "Par'}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": 'is", "unit": "c"}'}},
    {"type": "content_block_stop", "index": 1},
    {"type": "message_delta",
     "delta": {"stop_reason": "tool_use", "stop_sequence": None},
     "usage": {"output_tokens": 57}},
    {"type": "message_stop"},
]

def accumulate(events: list[dict]) -> dict:
    pass  # to complete

print(json.dumps(accumulate(RECORDED), indent=1))
# Expected: content = [text "Let me check the weather.",
#                      tool_use get_weather {"city": "Paris", "unit": "c"}]
#           stop_reason "tool_use", usage {"input_tokens": 1450, "output_tokens": 57}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def accumulate(events):
#     out = {"id": None, "model": None, "content": [], "stop_reason": None,
#            "usage": {"input_tokens": 0, "output_tokens": 0}}
#     acc = {}
#     for e in events:
#         t = e["type"]
#         if t == "message_start":
#             m = e["message"]
#             out["id"], out["model"] = m["id"], m["model"]
#             out["usage"]["input_tokens"] = m["usage"]["input_tokens"]
#         elif t == "content_block_start":
#             acc[e["index"]] = {"block": dict(e["content_block"]), "buf": []}
#         elif t == "content_block_delta":
#             d = e["delta"]
#             acc[e["index"]]["buf"].append(
#                 d.get("text") or d.get("thinking") or d.get("partial_json") or "")
#         elif t == "content_block_stop":
#             entry = acc[e["index"]]
#             block, raw = entry["block"], "".join(entry["buf"])
#             if block["type"] == "text":
#                 block["text"] = raw
#             elif block["type"] == "tool_use":
#                 block["input"] = json.loads(raw)      # only safe at block stop
#             entry["done"] = block
#         elif t == "message_delta":
#             out["stop_reason"] = e["delta"]["stop_reason"]
#             out["usage"]["output_tokens"] = e["usage"]["output_tokens"]
#     out["content"] = [acc[i]["done"] for i in sorted(acc)]
#     return out

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Stream latency metrics
# ════════════════════════════════════════════════════════
# TIMED is (elapsed_seconds, event_type) for one streamed response,
# t=0 being the moment the request was sent.
# Implement stream_metrics(timed, output_tokens) -> dict with:
#   "ttft"        : elapsed at the FIRST content_block_delta (time to first token)
#   "total"       : elapsed at message_stop
#   "generation"  : total - ttft
#   "tokens_per_s": output_tokens / generation, 0.0 if generation == 0
#   "stall_max"   : largest gap between two consecutive content_block_delta events
# Round every float to 3 decimals.

TIMED = [
    (0.000, "message_start"),
    (0.412, "content_block_start"),
    (0.418, "content_block_delta"),
    (0.455, "content_block_delta"),
    (0.902, "content_block_delta"),   # a stall — upstream hiccup
    (0.940, "content_block_delta"),
    (0.975, "content_block_stop"),
    (0.980, "message_delta"),
    (0.985, "message_stop"),
]

def stream_metrics(timed: list[tuple[float, str]], output_tokens: int) -> dict:
    pass  # to complete

print(stream_metrics(TIMED, output_tokens=57))
# Expected: {'ttft': 0.418, 'total': 0.985, 'generation': 0.567,
#            'tokens_per_s': 100.529, 'stall_max': 0.447}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def stream_metrics(timed, output_tokens):
#     deltas = [t for t, kind in timed if kind == "content_block_delta"]
#     ttft = deltas[0] if deltas else 0.0
#     total = next(t for t, kind in timed if kind == "message_stop")
#     generation = total - ttft
#     gaps = [b - a for a, b in zip(deltas, deltas[1:])]
#     return {
#         "ttft": round(ttft, 3),
#         "total": round(total, 3),
#         "generation": round(generation, 3),
#         "tokens_per_s": round(output_tokens / generation, 3) if generation else 0.0,
#         "stall_max": round(max(gaps), 3) if gaps else 0.0,
#     }

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Why you cannot parse tool JSON mid-stream
# ════════════════════════════════════════════════════════
# Implement parse_progress(fragments) -> (states, value) where
#   • states is a list of one character per fragment:
#       "x" if the buffer accumulated SO FAR is not valid JSON
#       "o" if it is
#   • value is the parsed object if the FULL buffer is valid, else None
# Then implement assemble_tool_input(fragments, closed) which returns the
# parsed dict when `closed` is True (content_block_stop was received) and
# raises ValueError("block not closed") otherwise — the discipline that
# prevents firing a tool with half its arguments.

FRAGMENTS = ['{"ci', 'ty": "Par', 'is", "unit"', ': "c"}']

def parse_progress(fragments: list[str]) -> tuple[list[str], Any]:
    pass  # to complete

def assemble_tool_input(fragments: list[str], closed: bool) -> dict:
    pass  # to complete

states, value = parse_progress(FRAGMENTS)
print("".join(states), "->", value)
print(assemble_tool_input(FRAGMENTS, closed=True))
try:
    assemble_tool_input(FRAGMENTS, closed=False)
except ValueError as e:
    print("refused:", e)
# Expected: xxxo -> {'city': 'Paris', 'unit': 'c'}
#           {'city': 'Paris', 'unit': 'c'}
#           refused: block not closed

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def parse_progress(fragments):
#     buf, states, value = "", [], None
#     for frag in fragments:
#         buf += frag
#         try:
#             value = json.loads(buf)
#             states.append("o")
#         except json.JSONDecodeError:
#             value = None
#             states.append("x")
#     return states, value
#
# def assemble_tool_input(fragments, closed):
#     if not closed:
#         raise ValueError("block not closed")
#     return json.loads("".join(fragments))

---
## 1.5 Structured output

### Lesson: Constrained decoding beats prompt-and-pray

**The intuition.** Asking a model to reply with only JSON in the system prompt is like asking a
chatty friend to answer only in one-word texts. Most of the time they will. Then one day they add
"sure, here you go!" before the word anyway, because a request is a suggestion, and people are
free to ignore suggestions. Constrained decoding isn't a politer version of the same request,
it's a different mechanism entirely: a form with pre-printed boxes, one character per box, where
the shape is enforced by the printing press, not by whoever's filling it in. The model isn't
being well-mannered about the format. It's structurally unable to write a token that breaks the
schema.

**The problem.** An agent is code, and code needs typed values, not prose. "Respond with only a JSON
object" in the system prompt is a *request*: it works 98% of the time, which in an agent that makes
20 model calls per run means roughly a third of runs contain a parse failure. Constrained decoding
makes the shape a property of the decoder rather than of the model's goodwill:
`output_config.format` with a JSON schema, or `strict: true` on a tool definition.

**Production pitfall.** Three that cost real time. (1) The old trick of prefilling the assistant turn
with `{` to force JSON is **removed**: an assistant prefill returns a 400 on Opus 5, Sonnet 5,
Fable 5 and the whole 4.6+ family. Any snippet you copy from a 2024 blog post will fail. (2) A strict
schema needs `"additionalProperties": false` **and** a complete `required` list; without them
"strict" silently means "mostly". (3) Constrained output is still cut off by `max_tokens`: you get
schema-valid-until-truncated JSON, so the `stop_reason` check from 1.2 stays mandatory. Also note
`output_config.format` is incompatible with citations (400).

**Where you meet it.** Every router that must return one of N labels, every extraction step feeding a
database, and the plan objects your agent emits in module 3.

**For the curious.** This isn't hypothetical. In February 2024, a Canadian tribunal held Air
Canada liable after its support chatbot invented a bereavement-fare policy that didn't exist; a
customer relied on it, and the airline's defense, that the chatbot was "a separate legal entity,"
did not go well. [CBC's writeup](https://www.cbc.ca/news/canada/british-columbia/air-canada-chatbot-lawsuit-1.7116416) has the details, and Vectara keeps a running catalogue of cases like it
at [awesome-agent-failures](https://github.com/vectara/awesome-agent-failures/blob/main/docs/case-studies/air-canada-chatbot-legal-ruling.md).

In [ ]:
# ── Pydantic-validated response (recommended) ────────────────
from pydantic import BaseModel, Field, ValidationError

class TriageResult(BaseModel):
    category: str = Field(description="billing | technical | account")
    severity: int = Field(ge=1, le=5)
    summary: str
    needs_human: bool

if LIVE:
    parsed = CLIENT.messages.parse(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user",
                   "content": "Ticket: card charged twice, customer furious, wants a callback."}],
        output_format=TriageResult,          # SDK derives the schema and validates the reply
    )
    result = parsed.parsed_output            # a real TriageResult instance
    print(result.category, result.severity, result.needs_human)
else:
    print("[offline] client.messages.parse(..., output_format=TriageResult).parsed_output")

# ── The raw wire form the SDK builds for you ─────────────────
raw_request = {
    "model": MODEL,
    "max_tokens": 1024,
    "messages": [{"role": "user", "content": "Ticket: card charged twice ..."}],
    "output_config": {"format": {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "category":    {"type": "string", "enum": ["billing", "technical", "account"]},
                "severity":    {"type": "integer"},
                "summary":     {"type": "string"},
                "needs_human": {"type": "boolean"},
            },
            "required": ["category", "severity", "summary", "needs_human"],
            "additionalProperties": False,      # mandatory for a strict schema
        },
    }},
}
print(json.dumps(raw_request["output_config"], indent=1)[:320])

# ── Strict tool arguments (same guarantee, tool side) ────────
strict_tool = {
    "name": "create_ticket",
    "description": "File a support ticket",
    "strict": True,                            # top-level, NOT inside tool_choice
    "input_schema": {
        "type": "object",
        "properties": {"title": {"type": "string"}, "priority": {"type": "integer"}},
        "required": ["title", "priority"],
        "additionalProperties": False,
    },
}

# ❌ Removed on every current model — a 400, not a fallback:
#    messages=[..., {"role": "assistant", "content": "{"}]

### Lesson: Defensive parsing for everything else

**The intuition.** Constrained decoding is the seatbelt in a brand-new car, but you won't only
ever drive brand-new cars. Older models, third-party gateways, and other agents in a pipeline
will all hand you "probably JSON" with no seatbelt fitted, which is why you still need a
defensive parser as the general-purpose airbag. Feeding the validation error back on retry is
really just good teaching. Hand a student's paper back marked only "wrong, try again" and you get
the same mistake twice. Circle the exact line and explain why it's wrong, and it actually gets
fixed. A model behaves the same way: it can correct an error it can read, not one it can only
guess at.

**The problem.** Constrained decoding does not cover every path you will meet: streamed partial
output, models called through a gateway that strips `output_config`, older models, and JSON produced
by *other* agents in a multi-agent system. For those you need a repair-and-validate stage that turns
"probably JSON" into a typed object or a clean, actionable failure.

**Production pitfall.** Two specifics. First, never do string matching on a serialised tool input:
Opus 5, Fable 5 and the 4.6+ family may escape Unicode or forward slashes differently, so
`'"city": "Paris"' in raw` is a coin flip while `json.loads(raw)["city"]` is not. Second, a bare retry
on a validation failure is a wasted call. Resend with the **validation error** in the message. Models
correct a schema violation they can read; they do not correct one they cannot see. Cap the attempts:
an uncapped repair loop is the classic runaway-cost incident (module 3 makes budgets explicit).

**Where you meet it.** Module 7 has you write `repair_json` from scratch under interview conditions;
module 5 needs it because sub-agent handoffs are JSON over a text channel.

In [ ]:
# ── The repair pipeline, stage by stage ──────────────────────
raw_outputs = [
    '{"category": "billing", "severity": 4}',                       # clean
    'Sure! Here you go:\n```json\n{"category": "billing"}\n```',   # fenced + preamble
    'The answer is {"category": "technical", "severity": 2} — hope that helps!',
]
for raw in raw_outputs:
    fenced = re.sub(r"^[^{]*```(?:json)?\s*|\s*```[^}]*$", "", raw.strip())
    start, end = fenced.find("{"), fenced.rfind("}")
    candidate = fenced[start:end + 1] if start != -1 and end != -1 else fenced
    try:
        print("ok  ->", json.loads(candidate))
    except json.JSONDecodeError as exc:
        print("fail->", exc.msg)

# ── Validate, then feed the error back ───────────────────────
def validation_feedback(exc: ValidationError) -> str:
    "Turn a pydantic error into an instruction the model can act on."
    lines = [f"- {'.'.join(str(p) for p in e['loc'])}: {e['msg']}" for e in exc.errors()]
    return "Your JSON failed validation:\n" + "\n".join(lines) + "\nReturn corrected JSON only."

try:
    TriageResult.model_validate({"category": "billing", "severity": 9, "summary": "x"})
except ValidationError as exc:
    print(validation_feedback(exc))

# ── Never regex a serialised tool input ──────────────────────
serialised = '{"path": "\\/var\\/log\\/app.log"}'   # legal JSON escaping of "/"
print("substring match:", '"/var/log/app.log"' in serialised)      # False — brittle
print("parsed         :", json.loads(serialised)["path"])          # /var/log/app.log

### 🏋️ Exercises 1.5

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Pydantic model -> strict JSON schema
# ════════════════════════════════════════════════════════
# The API accepts a strict schema only if EVERY object node declares
# "additionalProperties": false and lists all its properties in "required".
# Implement to_strict_schema(model_cls) -> dict that:
#   • starts from model_cls.model_json_schema()
#   • recursively, on every node with "type" == "object":
#       - sets "additionalProperties" to False
#       - sets "required" to the list of its property names, in declaration order
#   • recursively removes every "title" key (noise the API does not need)
#   • also walks "$defs", "items", and "properties" values
# Do not mutate the schema returned by pydantic.

class Address(BaseModel):
    city: str
    zipcode: str

class Customer(BaseModel):
    name: str
    address: Address
    tags: list[str]

def to_strict_schema(model_cls) -> dict:
    pass  # to complete

print(json.dumps(to_strict_schema(Customer), indent=1, sort_keys=True))
# Expected: no "title" anywhere; both the Customer node and the Address node
# in "$defs" carry "additionalProperties": false and a complete "required" list.

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# import copy
# def _strictify(node):
#     if isinstance(node, list):
#         return [_strictify(x) for x in node]
#     if not isinstance(node, dict):
#         return node
#     out = {k: _strictify(v) for k, v in node.items() if k != "title"}
#     if out.get("type") == "object":
#         out["additionalProperties"] = False
#         out["required"] = list(out.get("properties", {}).keys())
#     return out
#
# def to_strict_schema(model_cls):
#     return _strictify(copy.deepcopy(model_cls.model_json_schema()))

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — repair_json: survive free-text model output
# ════════════════════════════════════════════════════════
# Implement repair_json(text) -> dict | None applying, in order:
#   1. strip a markdown code fence (```json ... ``` or ``` ... ```)
#   2. extract the FIRST balanced {...} object — brace counting that ignores
#      braces inside strings and honours backslash escapes
#   3. drop trailing commas before } or ]
#   4. json.loads the result; return None on failure (never raise)
# A truncated object (never closed) must return None, not a partial dict.

CASES = [
    '{"a": 1, "b": [2, 3]}',
    '```json\n{"a": 1}\n```',
    'Here you go: {"a": 1, "note": "use {braces} freely"} — done!',
    '{"a": 1, "b": [2, 3,],}',
    '{"a": 1, "b": ',
    'no json at all',
]

def repair_json(text: str):
    pass  # to complete

for case in CASES:
    print(repr(case[:34]), "->", repair_json(case))
# Expected: {'a': 1, 'b': [2, 3]} / {'a': 1} / {'a': 1, 'note': 'use {braces} freely'}
#           / {'a': 1, 'b': [2, 3]} / None / None

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def _first_object(text):
#     start = text.find("{")
#     if start == -1:
#         return None
#     depth, in_string, escaped = 0, False, False
#     for i in range(start, len(text)):
#         ch = text[i]
#         if in_string:
#             if escaped:
#                 escaped = False
#             elif ch == "\\":
#                 escaped = True
#             elif ch == '"':
#                 in_string = False
#             continue
#         if ch == '"':
#             in_string = True
#         elif ch == "{":
#             depth += 1
#         elif ch == "}":
#             depth -= 1
#             if depth == 0:
#                 return text[start:i + 1]
#     return None                      # never closed -> truncated
#
# def repair_json(text):
#     cleaned = re.sub(r"```(?:json)?", "", text).strip()
#     candidate = _first_object(cleaned)
#     if candidate is None:
#         return None
#     candidate = re.sub(r",(\s*[}\]])", r"\1", candidate)   # trailing commas
#     try:
#         return json.loads(candidate)
#     except json.JSONDecodeError:
#         return None

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Validation retry that feeds the error back
# ════════════════════════════════════════════════════════
# Implement parse_with_retry(send, model_cls, max_attempts=3) -> (instance, attempts)
#   • send(feedback: str | None) -> str returns the model's raw text;
#     feedback is None on the first attempt, otherwise the validation error text
#   • repair with repair_json, then validate with model_cls.model_validate
#   • on ValidationError, retry with validation_feedback(exc) as the feedback
#   • if repair_json returns None, retry with the feedback
#     "Output was not valid JSON. Return a single JSON object only."
#   • after max_attempts, raise RuntimeError("gave up after N attempts")
#   • return (validated_instance, attempts_used)

SCRIPTED = [
    'sure, here: {"category": "billing", "severity": 9, "summary": "x"}',  # severity out of range
    'oops',                                                               # not JSON at all
    '{"category": "billing", "severity": 4, "summary": "double charge", "needs_human": true}',
]

def make_sender(scripted):
    seen = []
    def send(feedback):
        seen.append(feedback)
        return scripted[len(seen) - 1]
    return send, seen

def parse_with_retry(send, model_cls, max_attempts: int = 3):
    pass  # to complete

send, seen = make_sender(SCRIPTED)
instance, attempts = parse_with_retry(send, TriageResult)
print(instance)
print("attempts:", attempts)
print("feedback on attempt 2 line 1:", (seen[1] or "").splitlines()[0])
# Expected: category='billing' severity=4 summary='double charge' needs_human=True
#           attempts: 3
#           feedback on attempt 2 line 1: Your JSON failed validation:

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def parse_with_retry(send, model_cls, max_attempts=3):
#     feedback = None
#     for attempt in range(1, max_attempts + 1):
#         raw = send(feedback)
#         data = repair_json(raw)
#         if data is None:
#             feedback = "Output was not valid JSON. Return a single JSON object only."
#             continue
#         try:
#             return model_cls.model_validate(data), attempt
#         except ValidationError as exc:
#             feedback = validation_feedback(exc)
#     raise RuntimeError(f"gave up after {max_attempts} attempts")

---
## 1.6 Prompt caching

### Lesson: Prefix matching is the whole model

**The intuition.** Remember the amnesiac consultant from 1.1, the one who needs the whole case
file re-handed to them every call? Prompt caching is the bookmark you leave in that file. As long
as every page before the bookmark matches last time byte for byte, the consultant flips straight
to it instead of starting over at page one. That's where the 90% discount comes from. Bookmarks
are unforgiving, though. Scribble one word on page 3 and they can no longer trust anything after
it, so they read the whole file again from the start. That's the whole reason it's called a
prefix match. Almost the same file buys you nothing. Only identical up to here does.

**The problem.** An agent resends its entire context every turn: system prompt, tool definitions,
retrieved documents, full history. By turn 15 that is 100k+ tokens re-uploaded and re-processed for a
200-token question. Prompt caching lets the server keep the processed prefix, cutting cost by ~90% and
latency substantially on the cached portion. For a long-running agent it is not an optimisation, it is
the difference between viable and not.

**The one invariant.** Caching is a **prefix match**. The cache key is the exact bytes of the rendered
prompt up to each `cache_control` breakpoint, and the render order is fixed:
`tools` → `system` → `messages`. One changed byte at position N invalidates every breakpoint at or
after N. Design the prompt so stability decreases monotonically: frozen content first, volatile
content last.

**Production pitfall.** The minimum cacheable prefix is model-dependent **and not monotonic across
generations**: 512 tokens on Opus 5 / Fable 5, 1024 on Opus 4.8 and Sonnet 5, 2048 on Opus 4.7, 4096
on Opus 4.6 and Haiku 4.5. Below the minimum nothing caches. No error, just
`cache_creation_input_tokens: 0` forever. A prompt that cached fine on Opus 5 silently stops caching
when a config change points it at Haiku. Always verify with `usage`, never assume.

**Where you meet it.** Module 4's RAG pipeline (retrieved documents are the classic cacheable block),
module 5's sub-agents (a fork that rebuilds `system` misses the parent's cache entirely), and the cost
dashboard in module 6.

**For the curious.** [Anthropic's prompt caching docs](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) go deeper on breakpoint placement and TTL economics than this lesson
does, with the exact per-model pricing multipliers.

In [ ]:
# ── Where the breakpoint goes ────────────────────────────────
request = {
    "model": MODEL,
    "max_tokens": DEMO_MAX_TOKENS,
    "tools": [],                                   # renders FIRST (position 0)
    "system": [
        {"type": "text", "text": "You are a contract analyst."},          # frozen
        {"type": "text", "text": "<50k tokens of policy documents>",
         "cache_control": {"type": "ephemeral"}},   # breakpoint: caches tools + system
    ],
    "messages": [{"role": "user", "content": "Does clause 7 permit sublicensing?"}],  # volatile
}
print(json.dumps(request["system"], indent=1)[:300])

# ── TTL and economics ────────────────────────────────────────
# write 1.25x base input for the default 5-minute ttl, 2.00x for "1h"
# read  0.10x base input
# break-even: 2 requests at 5m ttl, 3 requests at 1h ttl
long_lived = {"type": "ephemeral", "ttl": "1h"}

MIN_CACHEABLE = {          # tokens; below this nothing caches, silently
    "claude-opus-5":    512,
    "claude-fable-5":   512,
    "claude-opus-4-8": 1024,
    "claude-sonnet-5": 1024,
    "claude-opus-4-7": 2048,
    "claude-opus-4-6": 4096,
    "claude-haiku-4-5": 4096,
}
for name, minimum in MIN_CACHEABLE.items():
    print(f"{name:<18} min cacheable prefix: {minimum:>5} tokens")

# ── Verify, always ───────────────────────────────────────────
resp = call(**request)
if resp is not None:
    u = resp.usage
    print("write:", u.cache_creation_input_tokens,
          "| read:", u.cache_read_input_tokens,
          "| uncached:", u.input_tokens)
    # read == 0 across repeated identical-prefix calls => a silent invalidator (next lesson)

### Lesson: Silent invalidators and the escape hatches

**The intuition.** Every invalidator on this list is the same mistake wearing a different coat:
something scribbles on a page before the bookmark and nobody notices. A timestamp in the system
prompt is a librarian stamping today's date on page 1 every time the book opens, so the bookmark
can never hold, because page 1 is never the same book twice. Unsorted JSON is a library that
reshelves the same books in a different order every night: identical contents, unrecognizable
sequence. And firing five identical requests at the same instant is like mailing five copies of
one letter to five different clerks at once, each one assuming somebody else already filed it.
Nobody has actually finished the original yet, so all five get treated as brand new.

**The problem.** Caching fails quietly. There is no warning, no error field, just a bill that never
goes down. Debugging it means diffing the rendered bytes of two consecutive requests and finding the
one that moved.

**Production pitfall.** The usual suspects, in rough order of frequency: `datetime.now()` interpolated
into the system prompt ("Today is 2026-08-19 14:32:07"), so the prefix changes every second;
`json.dumps(config)` without `sort_keys=True`, or anything serialised from a `set`, giving
non-deterministic byte order; a per-user or feature-flagged tool list, which sits at position 0 and so
invalidates *everything*; and conditional system sections (`if beta_user: system += ...`) that fork
your cache into 2^n distinct prefixes.

Two subtler ones. **Concurrent fan-out**: N identical requests fired simultaneously all miss, because
an entry becomes readable only once the first response *starts streaming*, so send one, await the first
token, then fire the rest. **The 20-block lookback**: a breakpoint searches backwards at most 20
content blocks for a prior entry, so an agent turn that appends 30 tool_use/tool_result blocks
silently loses the trail; add an intermediate breakpoint every ~15 blocks.

Not every change costs everything. Changing `tool_choice`, toggling thinking, or attaching images
invalidates only the messages tier; changing the system prompt keeps the tools tier; changing tools or
the model rebuilds all three. And there are two escape hatches: a mid-conversation
`{"role": "system"}` message instead of editing top-level `system` (1.1), and `tool_addition` /
`tool_removal` blocks instead of editing `tools`.

**Where you meet it.** Every "our LLM bill tripled after the refactor" postmortem you will ever read.

In [ ]:
# ── The invalidation hierarchy ───────────────────────────────
#   change                          tools   system  messages
#   tool definitions / model         ✗       ✗        ✗       (full rebuild)
#   system prompt content            ✓       ✗        ✗
#   tool_choice / thinking / images  ✓       ✓        ✗
#   new message content              ✓       ✓        ✗       (expected, harmless)

# ── ❌ A prefix that can never be reused ─────────────────────
def build_system_bad(user):
    return (f"Today is {time.strftime('%Y-%m-%d %H:%M:%S')}. "     # changes every second
            f"User: {user['id']}. "                                # per-user fork
            f"Config: {json.dumps(user['prefs'])}")                # unsorted keys

# ── ✅ Frozen prefix, volatile content pushed to the end ─────
SYSTEM_FROZEN = "You are a contract analyst. Cite clause numbers."

def build_request_good(user, question):
    return {
        "model": MODEL,
        "max_tokens": DEMO_MAX_TOKENS,
        "system": [{"type": "text", "text": SYSTEM_FROZEN,
                    "cache_control": {"type": "ephemeral"}}],
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": f"Context: user={user['id']} "
                                         f"prefs={json.dumps(user['prefs'], sort_keys=True)}"},
                {"type": "text", "text": question},          # after the breakpoint
            ]},
        ],
    }

user = {"id": "u_42", "prefs": {"lang": "fr", "verbosity": "low"}}
a = build_request_good(user, "Clause 7?")
b = build_request_good(user, "Clause 9?")
print("system identical across requests:", a["system"] == b["system"])

# ── Pre-warming: pay the write before traffic arrives ────────
prewarm = {
    "model": MODEL,
    "max_tokens": 0,                     # prefill only: content=[], no output billed
    "system": [{"type": "text", "text": SYSTEM_FROZEN,
                "cache_control": {"type": "ephemeral"}}],
    "messages": [{"role": "user", "content": "warmup"}],
}
# Worth it when first-request latency is user-visible AND the prefix is large AND
# there is a quiet moment before traffic (startup, deploy). Not for steady traffic.
print(json.dumps(prewarm, indent=1)[:200])

### 🏋️ Exercises 1.6

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Place cache breakpoints at stability boundaries
# ════════════════════════════════════════════════════════
# Implement plan_breakpoints(segments, min_cacheable, max_breakpoints=4) -> list[str]
# returning the names of the segments AFTER which to attach cache_control.
# `segments` are given in render order, each {"name", "tokens", "stability"}
# with stability in ("static", "per_session", "per_turn", "per_request").
# Rules:
#   • a candidate is the LAST segment of each contiguous stability group
#   • never place a breakpoint on a "per_request" group — it is never reused
#   • a candidate is only valid if the cumulative token count of the prefix
#     up to and including it is >= min_cacheable
#   • the API allows at most max_breakpoints per request: if there are more
#     valid candidates, keep the LAST ones (deepest prefixes)

SEGMENTS = [
    {"name": "tools",          "tokens":  900, "stability": "static"},
    {"name": "system_core",    "tokens": 1500, "stability": "static"},
    {"name": "persona",        "tokens":  300, "stability": "per_session"},
    {"name": "retrieved_docs", "tokens": 4000, "stability": "per_session"},
    {"name": "history",        "tokens": 2500, "stability": "per_turn"},
    {"name": "question",       "tokens":   80, "stability": "per_request"},
]

def plan_breakpoints(segments: list[dict], min_cacheable: int,
                     max_breakpoints: int = 4) -> list[str]:
    pass  # to complete

print(plan_breakpoints(SEGMENTS, min_cacheable=512))    # Opus 5
print(plan_breakpoints(SEGMENTS, min_cacheable=4096))   # Opus 4.6 / Haiku 4.5
# Expected: ['system_core', 'retrieved_docs', 'history']
#           ['retrieved_docs', 'history']

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def plan_breakpoints(segments, min_cacheable, max_breakpoints=4):
#     candidates, cumulative = [], 0
#     for i, seg in enumerate(segments):
#         cumulative += seg["tokens"]
#         is_last_of_group = (i == len(segments) - 1
#                             or segments[i + 1]["stability"] != seg["stability"])
#         if not is_last_of_group or seg["stability"] == "per_request":
#             continue
#         if cumulative >= min_cacheable:
#             candidates.append(seg["name"])
#     return candidates[-max_breakpoints:]

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Does caching actually pay off here?
# ════════════════════════════════════════════════════════
# Implement cache_economics(n, prefix_tokens, suffix_tokens, output_tokens,
#                           price_in, price_out, ttl="5m") -> dict with
#   "uncached", "cached", "saved" (dollars, rounded to 6 decimals) and
#   "break_even": the smallest number of requests at which cached < uncached.
# Model:
#   • uncached: every request pays (prefix + suffix) at price_in and output at price_out
#   • cached  : request 1 writes the prefix at price_in * write_mult
#               (1.25 for "5m", 2.00 for "1h"); requests 2..n read it at price_in * 0.10;
#               suffix and output always cost full price
# Prices are $ per 1M tokens. break_even may exceed n.

def cache_economics(n: int, prefix_tokens: int, suffix_tokens: int, output_tokens: int,
                    price_in: float, price_out: float, ttl: str = "5m") -> dict:
    pass  # to complete

print(cache_economics(10, 50_000, 500, 400, 5.00, 25.00))
print(cache_economics(10, 50_000, 500, 400, 5.00, 25.00, ttl="1h"))
# Expected: {'uncached': 2.625, 'cached': 0.6625, 'saved': 1.9625, 'break_even': 2}
#           {'uncached': 2.625, 'cached': 0.85, 'saved': 1.775, 'break_even': 3}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def cache_economics(n, prefix_tokens, suffix_tokens, output_tokens,
#                     price_in, price_out, ttl="5m"):
#     write_mult = {"5m": 1.25, "1h": 2.00}[ttl]
#
#     def uncached_cost(k):
#         return (k * (prefix_tokens + suffix_tokens) * price_in
#                 + k * output_tokens * price_out) / 1_000_000
#
#     def cached_cost(k):
#         prefix = prefix_tokens * price_in * (write_mult + 0.10 * (k - 1))
#         return (prefix + k * suffix_tokens * price_in
#                 + k * output_tokens * price_out) / 1_000_000
#
#     break_even = next((k for k in range(1, 1000) if cached_cost(k) < uncached_cost(k)), None)
#     return {"uncached": round(uncached_cost(n), 6),
#             "cached": round(cached_cost(n), 6),
#             "saved": round(uncached_cost(n) - cached_cost(n), 6),
#             "break_even": break_even}

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Find which cache tier a change destroyed
# ════════════════════════════════════════════════════════
# The rendered prefix is, in order: tools, then system, then messages.
# Implement diagnose(req_a, req_b) -> (tier, offset) where
#   • render(req) = json.dumps(req["tools"]) + json.dumps(req["system"])
#                                            + json.dumps(req["messages"])
#   • offset is the index of the first differing character between the two
#     rendered strings (or the length of the shorter one if it is a prefix of
#     the other); None if identical
#   • tier is the section that offset falls in: "tools", "system" or "messages",
#     and "none" when the renders are identical
# Losing "tools" means the whole request is uncached; "system" keeps the tools
# tier; "messages" is the normal, harmless case.

base = {
    "tools": [{"name": "search", "description": "search docs"}],
    "system": [{"type": "text", "text": "You are an analyst."}],
    "messages": [{"role": "user", "content": "clause 7?"}],
}
changed_msg    = {**base, "messages": [{"role": "user", "content": "clause 9?"}]}
changed_system = {**base, "system": [{"type": "text", "text": "You are an auditor."}]}
changed_tools  = {**base, "tools": [{"name": "search", "description": "search all docs"}]}

def diagnose(req_a: dict, req_b: dict) -> tuple[str, int | None]:
    pass  # to complete

for label, other in [("same", base), ("msg", changed_msg),
                     ("system", changed_system), ("tools", changed_tools)]:
    print(f"{label:<7}", diagnose(base, other))
# Expected: same    ('none', None)
#           msg     ('messages', 136)
#           system  ('system', 89)
#           tools   ('tools', 43)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def _sections(req):
#     tools = json.dumps(req["tools"])
#     system = json.dumps(req["system"])
#     messages = json.dumps(req["messages"])
#     return tools, system, messages
#
# def diagnose(req_a, req_b):
#     a_parts, b_parts = _sections(req_a), _sections(req_b)
#     a, b = "".join(a_parts), "".join(b_parts)
#     if a == b:
#         return "none", None
#     offset = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
#     bounds = [("tools", len(a_parts[0])),
#               ("system", len(a_parts[0]) + len(a_parts[1])),
#               ("messages", len(a))]
#     return next(name for name, end in bounds if offset < end), offset

---
## 📋 Module 1 Recap

| Topic | Test |
|---|---|
| Roles & authority | Keep untrusted data in `user`, operator instructions in `system` |
| Content blocks | Read a response without assuming `content[0].text` |
| Transcript rules | `messages[0]` is `user`; merge consecutive same-role turns |
| Multi-turn state | Append `response.content` whole: thinking, tool_use, compaction blocks survive |
| Mid-conversation system | Place a `role: "system"` message legally, and know why it beats editing `system` |
| Sampling mechanics | Softmax with temperature + nucleus filter, from scratch |
| Param compatibility | Strip `temperature`/`top_p`/`budget_tokens` for a current model instead of eating a 400 |
| Effort & thinking | Map a workload to `low` → `max`; adaptive thinking instead of `budget_tokens` |
| Stop handling | Branch on `stop_reason` before parsing; simulate `stop_sequences` + truncation |
| Token counting | `count_tokens` per model, never `tiktoken`, never `chars / 4` |
| Usage accounting | Prompt size = `input` + `cache_creation` + `cache_read`; build a run cost ledger |
| Context budgeting | Trim history to a budget without orphaning a turn |
| Chunking | Token-aware chunks with overlap and an injected counter |
| Stream protocol | Rebuild a full message from raw SSE events, per block index |
| Stream latency | TTFT, generation time, tokens/s, worst stall |
| Partial JSON | Accumulate `input_json_delta`, parse only at `content_block_stop` |
| Structured output | `messages.parse` with Pydantic; strict schema needs `additionalProperties: false` |
| Defensive parsing | Fence stripping, balanced-brace extraction, trailing commas, truncation → `None` |
| Validation retry | Feed the validation error back, capped attempts |
| Cache prefix model | Breakpoints at stability boundaries, ≤ 4, above the model's minimum |
| Cache economics | Break-even at 2 requests (5m TTL) / 3 (1h TTL) |
| Cache debugging | Locate the first divergent byte and name the tier it destroyed |

**Next:** `module2.ipynb` covers tool definitions, the `tool_use` → `tool_result` loop, parallel calls,
`tool_choice`, and feeding tool failures back to the model.